# 这里是 -> **[本篇笔记博客](https://blog.algieba12.cn/llm03-know-more-about-transformer-lib/)**

In [ ]:

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -U transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [3]:
from transformers import AutoTokenizer

model_path = "/kaggle/input/qwen2.5/transformers/7b-instruct/1/"

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

text = "Transformer is amazing"
# 1. 编码 (Encode): 文本 -> 数字 ID
input_ids = tokenizer.encode(text)
print(f"原文: {text}")
print(f"数字 ID: {input_ids}")

# 2. 解码 (Decode): 数字 ID -> 文本
decoded_text = tokenizer.decode(input_ids)
print(f"还原: {decoded_text}")

原文: Transformer is amazing
数字 ID: [46358, 374, 7897]
还原: Transformer is amazing


In [4]:
messages = [
    {"role": "system", "content": "你是一个物理学家。"},
    {"role": "user", "content": "用一句话解释相对论。"}
]

# 自动应用聊天模板
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("模型实际看到的输入:\n", formatted_prompt)

模型实际看到的输入:
 <|im_start|>system
你是一个物理学家。<|im_end|>
<|im_start|>user
用一句话解释相对论。<|im_end|>
<|im_start|>assistant



In [5]:
from transformers import AutoModelForCausalLM
import torch

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",      # 关键！自动将模型切分到两张 T4 显卡上
    torch_dtype=torch.float16, # 使用半精度，节省显存
    trust_remote_code=True
)

print(f"模型加载成功！显存分布: {model.hf_device_map}")

`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-25 15:14:40.018087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769354080.308959      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769354080.395059      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769354081.059627      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769354081.059659      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769354081.059662      55

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


模型加载成功！显存分布: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [6]:
# 定义一个测试函数
def test_generation(temp, top_p, prompt_text):
    messages = [
        {"role": "system", "content": "你是一个前卫的科幻小说家。"},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    print(f"\n======== 设置: Temperature={temp}, Top_p={top_p} ========")
    
    try:
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=100,  # 限制长度，方便快速看结果
            temperature=temp,
            top_p=top_p,
            do_sample=True,      # 必须开启采样
            pad_token_id=tokenizer.eos_token_id
        )
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        # 只打印回答部分
        print(response.split('assistant')[-1].strip())
    except Exception as e:
        print(f"生成出错: {e}")



In [7]:
# 测试 Prompt：需要一点想象力的话题
prompt = "请用这三个词写一个微故事：量子、失恋、炒饭。"

# 1. 低温模式 (0.1)：严谨、死板
test_generation(temp=0.1, top_p=0.9, prompt_text=prompt)

# 2. 适中模式 (0.7)：正常、流畅
test_generation(temp=0.7, top_p=0.9, prompt_text=prompt)

# 3. 高温模式 (1.5)：疯狂、混乱
# 注意：可能会输出乱码或完全不通顺的句子
test_generation(temp=1.5, top_p=0.9, prompt_text=prompt)


======== 设置: Temperature=0.1, Top_p=0.9 ========
在量子世界里，时间与空间的概念变得模糊不清，而李明的世界也因为一段失败的恋情而变得一片混沌。他尝试着用量子纠缠理论来修复自己破碎的心灵，却意外地将自己从现实世界送入了一个平行宇宙。

在这个平行宇宙中，李明发现了一家特别的餐馆，这里的厨师是一位曾经的恋人，她正在为一位顾客准备一道特别的炒饭。这道炒饭不仅色香味俱全，还

======== 设置: Temperature=0.7, Top_p=0.9 ========
在量子世界的边缘，李明独自一人坐在一家不起眼的小餐馆里，面前是一碗普通的炒饭。他和女朋友分手的原因，是因为她沉迷于虚拟现实中的量子世界，而他却对现实世界充满了留恋。

就在几天前，他们最后一次争吵后，她告诉他：“我找到了真正的自我——一个穿梭在量子世界的探索者。”她留下了一盘未吃完的炒饭，然后消失在了虚拟现实中。

李明望着那盘炒饭，

======== 设置: Temperature=1.5, Top_p=0.9 ========
标题：时空泡饭

李明最近失恋了，每天只能煮一大锅泡饭来消磨时间。
 
这天李明正在做饭，他忽然接收到一条量子信号。他惊奇地发现那是自己失恋前女友的坐标位置。想到能与自己相爱过的人共度时光是多么美妙的事情，他便将自己煮了一锅泡饭送进了坐标传送器。
 
结果却是一锅炒饭。
 
李明百思不解。


In [8]:
prompt_2 = "请给一种不存在的颜色起个名字，并描述它的样子。"

# 1. 极窄采样 (0.01)：只选概率最高的那个词（近似贪婪搜索）
test_generation(temp=0.8, top_p=0.01, prompt_text=prompt_2)

# 2. 宽广采样 (0.95)：允许罕见词出现
test_generation(temp=0.8, top_p=0.95, prompt_text=prompt_2)


======== 设置: Temperature=0.8, Top_p=0.01 ========
这种不存在的颜色我命名为“星尘紫”。它是一种梦幻般的颜色，介于紫色和银色之间，仿佛是宇宙中无数微小的星辰碎片在闪烁时所散发出的光芒。在不同的光线下，星尘紫会呈现出不同的色调，有时偏紫，有时偏银，有时又像是掺杂了点点星光的淡蓝色。它既神秘又优雅，仿佛能让人感受到宇宙的浩瀚与深邃。

======== 设置: Temperature=0.8, Top_p=0.95 ========
这种不存在的颜色我称之为“星际幻彩”（Stellar Mirage）。在视觉上，它并非单一色调，而是一种动态变化的色彩组合，像是无数微小的光点在眼前闪烁变幻，这些光点包含了所有可见光谱的颜色，同时又带着一种神秘的、不可名状的色彩。

当观察者注视着“星际幻彩”的时候，他可以看到蓝、绿、紫等颜色快速地在眼前切换和混合，它们以一种几乎


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. 设置模型 ID
model_id = "/kaggle/input/qwen2.5/transformers/7b-instruct/1/"

# 2. 加载翻译官
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# 3. 加载大脑 (双卡模式)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# 4. 准备输入
prompt = "请用这三个词写一个微小说：Kaggle、深夜、爆显存"
messages = [
    {"role": "system", "content": "你是一个幽默的程序员。"},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 5. 生成 (调节参数！)
print(" 正在生成...")
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.8,  # 稍微有点创意
    top_p=0.9,        # 剔除离谱词
    do_sample=True    # 必须开启采样，温度才生效
)

# 6. 解码并输出
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("-" * 20)
print(f" 回答:\n{response}")
print("-" * 20)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


 正在生成...
--------------------
 回答:
在一个寒冷的深夜，李雷坐在他那间堆满咖啡罐和代码文件的房间里，屏幕上是Kaggle竞赛的数据集。他正尝试训练一个复杂的深度学习模型。然而，就在他认为胜利在望的时候，“嘶~”的一声，显示器瞬间变成了一片漆黑，伴随着一声悲壮的“爆显存了”。

李雷揉了揉眼睛，看着眼前一片空白的屏幕，心中充满了挫败感，但他转念一想：“还好不是‘爆内存了’，否则我这台老旧电脑可能就要彻底退休了。”于是，他又开始调整参数，希望能在这个深夜里找到那个隐藏在数据海洋中的宝藏。
--------------------
